In [ ]:
import re
from collections import Counter
import json

# Import your preferred deep learning framework here
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
# import torch.utils.data as data

In [ ]:
# Loading the training data from a JSON file
with open('train.json', 'r') as train_file:
    train_data = json.load(train_file)

print("Train data loaded:", len(train_data), "samples")
print(f"{train_data[0]}\n{train_data[1]}")

Train data loaded: 7401 samples
{'reviews': 'I bought this belt for my daughter in-law for Christmas and she loved it', 'sentiments': 1}
{'reviews': 'The size was perfect and so was the color.  It looked just like it did on the web page:)', 'sentiments': 1}


In [ ]:
# Tokenization to clean and split the text into words
def tokenize(text):
    text = text.lower()
    text = text.replace("’", "'") # replace fancy apostrophes with normal ones
    text = text.replace("‘", "'") # replace fancy opening apostrophes with normal ones
    text = text.replace("n't", "nt") # remove apostrophes from can't, won't, and keep it's, we'll, ect
    text = text.replace(r"(http)\S+", "") # remove URLs
    text = re.sub(r"([a-z])\1{2,}", r"\1", text) # soooo -> so
    return re.findall(r"[a-z0-9]+|[!?]", text) # remove punctuation except for ! and ?

# Counting the frequency of each token in the training data to build a vocabulary
MAX_VOCAB = 20000
counter = Counter()
for item in train_data:
    counter.update(tokenize(item['reviews']))
most_common = counter.most_common(MAX_VOCAB - 2)

vocab = {"<pad>": 0, "<unk>": 1}
for word, _ in most_common:
    vocab[word] = len(vocab)

vocab_size = min(len(vocab), MAX_VOCAB)
print("Vocabulary size:", vocab_size)

def encode(text):
    tokens = tokenize(text)
    return [vocab.get(t, 1) for t in tokens]  

encoded = [encode(item['reviews']) for item in train_data]

Vocabulary size: 14082


In [ ]:
# Percentile to determine MAX_LEN for padding
import numpy as np
lens = np.array([len(s) for s in encoded])
print("median:", np.median(lens), "90th pct:", np.percentile(lens, 90), "95th pct:", np.percentile(lens, 95))

median: 43.0 90th pct: 111.0 95th pct: 148.0


In [ ]:
MAX_LEN = 120 # since 90th percentile is 111, limit to 120 to be safe
padded = [(ids + [0] * MAX_LEN)[:MAX_LEN] for ids in encoded]

# X = torch.tensor(padded)
# y_true = torch.tensor([item['sentiments'] for item in train_data], dtype=torch.float32)
# print("X shape:", X.shape)
# print("y_true shape:", y_true.shape)

X shape: torch.Size([7401, 120])
y_true shape: torch.Size([7401])
